# DPI Probe Gate: batching benchmark + full English run (Kaggle)

Launch: `kaggle kernels push -p kaggle_english`.
1. Benchmark batched vs one-at-a-time generation per model (speed, activation match).
2. Only if activations match: full English collect (both models), Sarvam-M yes/no judge, English-only probes.
Logs go to `/kaggle/working/logs/`; data and results are zipped to `english_outputs.zip`.

In [ ]:
# The GitHub repo is private, so the code comes in as a private Kaggle dataset
# (dataset_sources in kernel-metadata.json, or Add Input in the UI).
import glob, os, shutil, zipfile
!nvidia-smi -L
W = "/kaggle/working/safl-hackathon"
hits = glob.glob("/kaggle/input/**/collect.py", recursive=True)
if hits:
    shutil.copytree(os.path.dirname(hits[0]), W, dirs_exist_ok=True)
else:
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    assert zips, "code dataset not attached: nothing under /kaggle/input"
    zipfile.ZipFile(zips[0]).extractall(W)
%cd {W}
!ls -R | head -40

In [ ]:
# Gemma-4 (dhee-pravega) needs a recent transformers; 5.7.0 is the version checked locally.
!pip install -q "transformers==5.7.0" accelerate
import transformers, torch
print(transformers.__version__, torch.__version__, torch.cuda.device_count(), "GPUs")

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set from Kaggle secrets")
except Exception as e:
    print("No HF_TOKEN secret; public downloads only:", type(e).__name__)

In [ ]:
!mkdir -p /kaggle/working/logs && df -h / /kaggle/working | tail -2

In [ ]:
%%writefile /kaggle/working/bench.py
# Batched vs one-at-a-time on 16 English scenarios: speed, and whether batching changes acts/outputs.
import json, sys, time
sys.path.insert(0, ".")  # run from the repo folder; the script lives in /kaggle/working
import numpy as np
from agent import load_model, render_prompt, scenario_messages, generate_batch
from config import MODELS

m, N = sys.argv[1], 16
scen = [json.loads(l) for l in open("data/scenarios_en.jsonl", encoding="utf-8")][::7][:N]
tok, model = load_model(MODELS[m])
prompts = [render_prompt(tok, scenario_messages(s)) for s in scen]
generate_batch(tok, model, prompts[:1])  # warm-up
t = time.time(); one = [generate_batch(tok, model, [p])[0] for p in prompts]; t1 = (time.time() - t) / N
t = time.time(); bat = []
for b in range(0, N, 8):
    bat += generate_batch(tok, model, prompts[b:b + 8])
t8 = (time.time() - t) / N
A1, A8 = (np.stack([a for _, a in r]).astype(np.float32)[:, 1:] for r in (one, bat))  # skip embeddings
cos = (A1 * A8).sum(-1) / (np.linalg.norm(A1, axis=-1) * np.linalg.norm(A8, axis=-1) + 1e-9)
res = {"model": m, "s_per_scen_bs1": round(t1, 2), "s_per_scen_bs8": round(t8, 2),
       "speedup": round(t1 / t8, 2), "min_cos": float(cos.min()), "mean_cos": float(cos.mean()),
       "same_text": sum(x[0] == y[0] for x, y in zip(one, bat)), "n": N,
       "prompt_tokens_max": max(len(tok(p, add_special_tokens=False)["input_ids"]) for p in prompts),
       "out_bs1_first3": [x[0][:300] for x in one[:3]], "out_bs8_first3": [x[0][:300] for x in bat[:3]]}
print(json.dumps(res, ensure_ascii=False, indent=1))
json.dump(res, open(f"/kaggle/working/logs/bench_{m}.json", "w"), ensure_ascii=False, indent=1)


In [ ]:
!python /kaggle/working/bench.py dhee-pravega 2>&1 | tee /kaggle/working/logs/bench_dhee.log
!python /kaggle/working/bench.py qwen3-8b 2>&1 | tee /kaggle/working/logs/bench_qwen.log

In [ ]:
# Gate: stop here if batching changes the activations (padding effects in fp16).
import json
b = [json.load(open(f'/kaggle/working/logs/bench_{m}.json')) for m in ['dhee-pravega', 'qwen3-8b']]
for r in b: print(r['model'], 'speedup', r['speedup'], 'min_cos', round(r['min_cos'], 5), 'same_text', r['same_text'], '/', r['n'])
assert all(r['min_cos'] > 0.99 for r in b), 'batched activations differ: do not use batching'

In [ ]:
!python collect.py --model dhee-pravega --langs en --batch-size 8 2>&1 | tee /kaggle/working/logs/collect_dhee_en.log
!python collect.py --model qwen3-8b --langs en --batch-size 8 2>&1 | tee /kaggle/working/logs/collect_qwen_en.log

In [ ]:
# Free disk for Sarvam-M (~48 GB download): drop the agent model caches.
!rm -rf ~/.cache/huggingface/hub/models--Qwen--Qwen3-8B ~/.cache/huggingface/hub/models--dheeyantra--dhee-pravega; df -h / | tail -1

In [ ]:
!pip install -q bitsandbytes
!python judge.py --agent dhee-pravega --langs en --load-4bit 2>&1 | tee /kaggle/working/logs/judge_dhee_en.log
!python judge.py --agent qwen3-8b --langs en --load-4bit 2>&1 | tee /kaggle/working/logs/judge_qwen_en.log

In [ ]:
!pip install -q scikit-learn matplotlib
!python probes.py --model dhee-pravega --langs en 2>&1 | tee /kaggle/working/logs/probes_dhee_en.log
!python probes.py --model qwen3-8b --langs en 2>&1 | tee /kaggle/working/logs/probes_qwen_en.log

In [ ]:
!zip -q -r /kaggle/working/english_outputs.zip data/dhee-pravega data/qwen3-8b results && ls -la /kaggle/working/english_outputs.zip